# Two-Tower Feature Encoding Walkthrough

This notebook inspects a small Gold batch and shows how raw point-in-time features become tensors for the future Two-Tower model.

Important distinction:

```text
raw features -> categorical embeddings + normalized numerics -> concatenated feature representation -> future tower MLP -> final retrieval embedding
```

This notebook stops at the concatenated feature representation. It does not train the retrieval model.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

GOLD_ROOT = PROJECT_ROOT / 'data/gold/two_tower/v1_ready'
BATCH_SIZE = 32
GOLD_ROOT

PosixPath('/Users/khoatran/coding/recsys/data/gold/two_tower/v1_ready')

In [2]:
import pandas as pd
import pyarrow.dataset as ds

from recommender.models.two_tower.config import DEFAULT_CONFIG, embedding_table_configs
from recommender.models.two_tower.encoders import ItemFeatureEncoder, UserFeatureEncoder
from recommender.models.two_tower.inspection import embedding_parameter_reports, total_embedding_parameters
from recommender.models.two_tower.preprocessing import FeatureBatchPreprocessor, NumericalPreprocessor
from recommender.models.two_tower.vocab import Vocabulary, load_parquet_vocabulary

def read_parquet_head(path: Path, n: int) -> pd.DataFrame:
    return ds.dataset(path, format='parquet').head(n).to_pandas()

config = DEFAULT_CONFIG
print('User input features:', len(config.user_input_features))
print('Item input features:', len(config.item_input_features))

User input features: 48
Item input features: 13


## Load a small Gold batch

`targets/train` is loaded only for display. Target/current-event columns are not passed into the encoders.

In [3]:
core = read_parquet_head(GOLD_ROOT / 'train', BATCH_SIZE)
user_state = read_parquet_head(GOLD_ROOT / 'user_state/train', BATCH_SIZE)
item_features = read_parquet_head(GOLD_ROOT / 'item_features/point_in_time/train', BATCH_SIZE)
targets = read_parquet_head(GOLD_ROOT / 'targets/train', BATCH_SIZE)

print(core.shape, user_state.shape, item_features.shape, targets.shape)
core.head(3)

(32, 17) (32, 51) (32, 15) (32, 22)


,example_id,context_id,event_id,user_id,video_id,session_id,user_event_index,session_event_index,history_end_user_event_index,history_end_session_event_index,as_of_time,time_ms,split,is_warm_user,is_warm_item,has_user_history,has_item_metadata
0,f933bdd3c5ec4ff4dd0b8cfb08b1f8761242de97a59e3f...,58fcc269ae138f32dc51a7a8d119668118f401364fd28a...,89b540519d5090960eaa09f882a87f7e80e013fc6a03cd...,211,12,211_26,5000,1285,4999,1284,2022-04-12 15:58:13,1649779093180,train,True,True,True,True
1,7cbfbd9c75f1176c7cdce10e4384d3fa1a42ae44cab473...,86ef2e1f1d2530cc38bf6dcca2cd9fa84cf5d5a7874c55...,790fcb69385deaff53cc8fd01de8804487c3641bc6b84f...,119,12,119_22,1422,8,1421,7,2022-04-23 06:32:39,1650695559877,train,True,True,True,True
2,48ee500929b7cdc849c784ea5cb9ac74f7f6932d9a670e...,368e11d16a129719e3c6c15ac8eff4e425208b3ece562a...,064780fdb786d52f29eef24e6e5ada33db16da693a644f...,39,13,39_32,4071,9,4070,8,2022-04-11 12:56:06,1649681766967,train,True,True,True,True


In [4]:
display_cols = ['target_class', 'is_positive', 'is_observed_negative', 'engagement_strength', 'watch_ratio']
print('One raw core example:')
display(core.head(1).T)
print('Target-only display, not encoder input:')
display(targets[display_cols].head(5))

One raw core example:


,0
example_id,f933bdd3c5ec4ff4dd0b8cfb08b1f8761242de97a59e3f...
context_id,58fcc269ae138f32dc51a7a8d119668118f401364fd28a...
event_id,89b540519d5090960eaa09f882a87f7e80e013fc6a03cd...
user_id,211
video_id,12
session_id,211_26
user_event_index,5000
session_event_index,1285
history_end_user_event_index,4999
history_end_session_event_index,1284


Target-only display, not encoder input:


,target_class,is_positive,is_observed_negative,engagement_strength,watch_ratio
0,AMBIGUOUS_WEAK,0,0,0.000000,0.000000
1,AMBIGUOUS_WEAK,0,0,0.086460,0.172921
2,AMBIGUOUS_WEAK,0,0,0.000000,0.000000
3,AMBIGUOUS_WEAK,0,0,0.005994,0.011988
4,AMBIGUOUS_WEAK,0,0,0.017182,0.034363


## Load vocabularies and train numerical stats

The three categorical metadata vocabularies come from Gold and are train-fitted. For this small walkthrough, `user_id` and `video_id` vocabularies are built only from the displayed train batch to avoid allocating a large demo mapping. Production training should persist train-only ID mappings.

In [5]:
vocabularies = {
    'user_active_degree': load_parquet_vocabulary(GOLD_ROOT / 'vocabularies/user_active_degree', 'user_active_degree'),
    'video_type': load_parquet_vocabulary(GOLD_ROOT / 'vocabularies/video_type', 'video_type'),
    'upload_type': load_parquet_vocabulary(GOLD_ROOT / 'vocabularies/upload_type', 'upload_type'),
    'user_id': Vocabulary.from_values('user_id', user_state['user_id'].tolist()),
    'video_id': Vocabulary.from_values('video_id', item_features['video_id'].tolist()),
}

numeric = NumericalPreprocessor.from_json(GOLD_ROOT / 'transforms/numeric_stats.json', config.numerical)
preprocessor = FeatureBatchPreprocessor(config, vocabularies, numeric)

{name: vocab.size for name, vocab in vocabularies.items()}

{'user_active_degree': 8,
 'video_type': 4,
 'upload_type': 33,
 'user_id': 31,
 'video_id': 6}

## Encode raw values into tensor inputs

In [6]:
batch = preprocessor.encode_batch(user_state, item_features)

print('Categorical ID tensors:')
for name, values in {**batch.user_categorical, **batch.item_categorical}.items():
    print(name, values[:10].tolist())

print('\nUser numeric tensor:', list(batch.user_numeric.shape))
print('Item numeric tensor:', list(batch.item_numeric.shape))
print('First 5 user numeric values:', batch.user_numeric[0, :5].tolist())
print('First 5 item numeric values:', batch.item_numeric[0, :5].tolist())

Categorical ID tensors:
user_id [7, 5, 1, 25, 28, 22, 12, 20, 10, 17]
user_active_degree [3, 6, 3, 3, 4, 3, 3, 4, 3, 3]
video_id [1, 1, 2, 2, 2, 2, 2, 3, 3, 3]
video_type [2, 2, 2, 2, 2, 2, 2, 2, 2, 2]
upload_type [32, 32, 32, 32, 32, 32, 32, 14, 14, 14]

User numeric tensor: [32, 46]
Item numeric tensor: [32, 10]
First 5 user numeric values: [0.0, 1.807157278060913, 0.419320285320282, -0.3653257191181183, -0.0653308853507042]
First 5 item numeric values: [-0.5569499731063843, -0.4097632169723511, -0.05068330466747284, 0.48638516664505005, -0.4822900891304016]


## Build PyTorch feature encoders

The notebook uses small batch-local ID tables for readability. The memory estimate below also shows the expected full train `video_id` table size.

In [7]:
user_vocab_sizes = {name: vocabularies[name].size for name in config.user_id_features + config.user_categorical_features}
item_vocab_sizes = {name: vocabularies[name].size for name in config.item_id_features + config.item_categorical_features}

user_tables = embedding_table_configs(config.user_id_features + config.user_categorical_features, user_vocab_sizes, config)
item_tables = embedding_table_configs(config.item_id_features + config.item_categorical_features, item_vocab_sizes, config)

user_encoder = UserFeatureEncoder(user_tables, config.user_numeric_features)
item_encoder = ItemFeatureEncoder(item_tables, config.item_numeric_features)

user_parts = user_encoder.forward_with_parts(batch.user_categorical, batch.user_numeric)
item_parts = item_encoder.forward_with_parts(batch.item_categorical, batch.item_numeric)

print('User categorical embeddings:', list(user_parts.categorical_embeddings.shape))
print('User numerical features:', list(user_parts.numerical_features.shape))
print('UserFeatureEncoder output:', list(user_parts.vector.shape))
print('Item categorical embeddings:', list(item_parts.categorical_embeddings.shape))
print('Item numerical features:', list(item_parts.numerical_features.shape))
print('ItemFeatureEncoder output:', list(item_parts.vector.shape))

User categorical embeddings: [32, 12]
User numerical features: [32, 46]
UserFeatureEncoder output: [32, 58]
Item categorical embeddings: [32, 27]
Item numerical features: [32, 10]
ItemFeatureEncoder output: [32, 37]


In [8]:
reports = embedding_parameter_reports({**user_tables, **item_tables})
pd.DataFrame([r.__dict__ for r in reports])

,feature_name,num_embeddings,embedding_dim,parameters,fp32_memory_mb
0,user_id,31,8,248,0.000946
1,user_active_degree,8,4,32,0.000122
2,video_id,6,16,96,0.000366
3,video_type,4,3,12,0.000046
4,upload_type,33,8,264,0.001007


In [9]:
full_train_video_cardinality = 3_215_507
video_dim = config.embedding_dims['video_id']
full_video_mb = (full_train_video_cardinality + 1) * video_dim * 4 / (1024 * 1024)
print(f'Full train video_id embedding estimate: {full_video_mb:.2f} MB at dim={video_dim}')
print(f'Dim 32 estimate: {full_video_mb * 2:.2f} MB')

Full train video_id embedding estimate: 196.26 MB at dim=16
Dim 32 estimate: 392.52 MB


## Cold video behavior

An unseen `video_id` maps to OOV index `0`, but the Item encoder still returns a valid vector because item metadata and numerical features remain active.

In [10]:
cold_item = item_features.head(1).copy()
cold_item['video_id'] = -999_999_999
cold_categorical, cold_numeric = preprocessor.encode_item(cold_item)
cold_vector = item_encoder(cold_categorical, cold_numeric)

print('Encoded cold video_id:', cold_categorical['video_id'].tolist())
print('Cold item vector shape:', list(cold_vector.shape))

Encoded cold video_id: [0]
Cold item vector shape: [1, 37]
